# Mean-Reversion Analysis — v5-aligned evaluation

This notebook is the **AR/OU mean-reversion benchmark** for the thesis, scored on the
**identical protocol** as `model_htboost_v5.ipynb` (the source of truth) via the shared
`thesis_eval` module. It produces, on the Norwegian swap tenors:

1. A **continuous out-of-sample forecast** of the h-day rate *change* from an
   **Ornstein–Uhlenbeck / AR(1)** model refit on each fold's training window.
2. The **shared metrics long-CSV** (`mean_reversion_metrics_long.csv`) that merges with
   `v5_metrics_long.csv` with zero reconciliation.
3. A **trading-sim export** (`mean_reversion_oos_predictions.csv`) of causal walk-forward
   OOS predictions with positions and a threshold-ready `yhat` / `train_pred_std`.

**Protocol (from `thesis_eval`, copied from v5):** walk-forward expanding-window regime
folds (GFC, ZIRP×2, COVID, Hiking); target `y = level.shift(-h) - level`; label-overlap
purge of `H-1` business days at every train/test boundary; horizons `H ∈ {1,5,21,63}`;
Random-walk benchmark (`ŷ_RW = 0`); Campbell–Thompson OOS-R², Clark–West,
Diebold–Mariano–Harvey, Pesaran–Timmermann, one-sided binomial; Bonferroni + BH-FDR within
the walk-forward `{horizon × tenor × regime}` family.

> The full-sample ADF / KPSS / OU half-life tests below are **descriptive motivation only** —
> they are computed on the whole sample and are *not* evidence of OOS skill. All headline
> numbers come from the causal walk-forward sweep.

## References — literature motivating each part

**Mean reversion / term-structure dynamics (descriptive screen & model spec).**
- Uhlenbeck, G. E., & Ornstein, L. S. (1930). On the Theory of the Brownian Motion. *Physical Review*, 36(5), 823–841.
- Vasicek, O. (1977). An Equilibrium Characterization of the Term Structure. *Journal of Financial Economics*, 5(2), 177–188.
- Cox, J. C., Ingersoll, J. E., & Ross, S. A. (1985). A Theory of the Term Structure of Interest Rates. *Econometrica*, 53(2), 385–407.
- Fama, E. F., & Bliss, R. R. (1987). The Information in Long-Maturity Forward Rates. *American Economic Review*, 77(4), 680–692.
- Campbell, J. Y., & Shiller, R. J. (1991). Yield Spreads and Interest Rate Movements: A Bird's Eye View. *Review of Economic Studies*, 58(3), 495–514.
- Cochrane, J. H., & Piazzesi, M. (2005). Bond Risk Premia. *American Economic Review*, 95(1), 138–160.
- Poterba, J. M., & Summers, L. H. (1988). Mean Reversion in Stock Prices: Evidence and Implications. *Journal of Financial Economics*, 22(1), 27–59.
- Balvers, R., Wu, Y., & Gilliland, E. (2000). Mean Reversion across National Stock Markets and Parametric Contrarian Investment Strategies. *Journal of Finance*, 55(2), 745–772.

**Stationarity diagnostics (motivation only, not OOS evidence).**
- Dickey, D. A., & Fuller, W. A. (1979). Distribution of the Estimators for Autoregressive Time Series with a Unit Root. *Journal of the American Statistical Association*, 74(366), 427–431.
- Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., & Shin, Y. (1992). Testing the Null Hypothesis of Stationarity against the Alternative of a Unit Root (KPSS). *Journal of Econometrics*, 54(1–3), 159–178.

**Out-of-sample evaluation, multiple testing & cross-validation (shared harness).**
- Campbell, J. Y., & Thompson, S. B. (2008). Predicting Excess Stock Returns Out of Sample: Can Anything Beat the Historical Average? *Review of Financial Studies*, 21(4), 1509–1531.
- Welch, I., & Goyal, A. (2008). A Comprehensive Look at the Empirical Performance of Equity Premium Prediction. *Review of Financial Studies*, 21(4), 1455–1508.
- Clark, T. E., & West, K. D. (2007). Approximately Normal Tests for Equal Predictive Accuracy in Nested Models. *Journal of Econometrics*, 138(1), 291–311.
- Diebold, F. X., & Mariano, R. S. (1995). Comparing Predictive Accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Harvey, D., Leybourne, S., & Newbold, P. (1997). Testing the Equality of Prediction Mean Squared Errors. *International Journal of Forecasting*, 13(2), 281–291.
- Pesaran, M. H., & Timmermann, A. (1992). A Simple Nonparametric Test of Predictive Performance. *Journal of Business & Economic Statistics*, 10(4), 461–465.
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the False Discovery Rate: A Practical and Powerful Approach to Multiple Testing. *Journal of the Royal Statistical Society: Series B*, 57(1), 289–300.
- Harvey, C. R., Liu, Y., & Zhu, H. (2016). ... and the Cross-Section of Expected Returns. *Review of Financial Studies*, 29(1), 5–68.
- López de Prado, M. (2018). *Advances in Financial Machine Learning*. Wiley (purged K-fold + embargo).
- Bergmeir, C., Hyndman, R. J., & Koo, B. (2018). A Note on the Validity of Cross-Validation for Evaluating Autoregressive Time Series Prediction. *Computational Statistics & Data Analysis*, 120, 70–83.

## Pre-registered decision rule (stated before running)

This is a **null-confirmation harness** (mirroring `model_htboost_v5.ipynb`). We commit,
*ex ante*, to the following standard for declaring genuine mean-reversion skill:

1. **Multiple testing.** A `(tenor × horizon × regime)` cell counts only if its one-sided
   binomial p-value clears **Bonferroni** *or* **BH-FDR** correction (Benjamini & Hochberg,
   1995; Harvey, Liu & Zhu, 2016) over the full walk-forward family `N` — recomputed, and
   **never pooled** across the NOR and full-universe families.
2. **Multiple regimes.** Positive OOS directional accuracy must hold across **multiple**
   regimes (GFC, ZIRP, COVID, Hiking), not a single one. One-regime strength is not a lead.
3. **Null confirmation.** OOS `dir_acc ≈ 0.50` together with a **negative Campbell–Thompson
   OOS-R²** (`ct_r2_oos < 0` — the model loses to the random walk; Campbell & Thompson, 2008)
   **confirms the null**. That is the designed, expected outcome; the train-vs-OOS gap is the
   finding, not a failure.

Because the h-day labels overlap, the directional tests are inflated. We therefore report the
**effective sample size** `n_eff ≈ n/H` next to every `dir_acc`; the binomial and
Pesaran–Timmermann (1992) p-values must be read against `n_eff`, not the raw `n`.

In [ ]:
import sys, os
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning, ValueWarning

sys.path.insert(0, '.')
from data_loader import load_data, MATURITY_NAMES
import thesis_eval as te

pd.set_option('display.max_rows', 120)
# Narrowed from a blanket ignore so genuine issues surface during the handoff run;
# keep only the known-benign numerical/statsmodels warnings the descriptive screens emit.
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=InterpolationWarning)
warnings.filterwarnings('ignore', category=ValueWarning)

# ── Identity / provenance (stamped on every metrics row) ─────────────────────
NOTEBOOK   = 'mean_reversion'
MODEL_KIND = 'mean_reversion'
RUN_TS     = pd.Timestamp.now(tz='UTC').isoformat()

# ── Headline universe + horizons come from the shared module (same as v5) ────
UNIVERSE   = te.NOR_TENORS          # NOR_1Y, NOR_5Y, NOR_10Y, NOR_15Y, NOR_30Y
H_GRID     = te.H_GRID              # [1, 5, 21, 63]
WF_FOLDS   = te.WF_FOLDS
MIN_TRAIN_OBS, MIN_TEST_OBS = te.MIN_TRAIN_OBS, te.MIN_TEST_OBS

# ── Frozen model config (pre-committed; nothing chosen on OOS data) ──────────
# Pure OU/AR(1): the daily mean-reversion speed (phi) and long-run level (mu) are
# estimated by OLS on each fold's TRAINING levels only — there is NO window/threshold
# hyperparameter to tune, so the model is leakage-free by construction.
FROZEN_CFG  = dict(model='ou_ar1', target='level_change', target_lags=1)
CONFIG_HASH = te.config_hash(FROZEN_CFG, extra='ou_ar1_levelchg')
FEATURE_COUNT = 1                   # the OU deviation (level - mu) drives the forecast

# ── Trading-export conventions (STEP 5) ──────────────────────────────────────
# Task convention: a long position (+1) = fixed-rate payer, and a predicted RATE
# DECREASE -> long (+1), predicted increase -> short (-1).  => position = -sign(yhat).
# (NB this is the literal task instruction; it is the OPPOSITE sign of swap_PL.ipynb's
#  payer=+1-on-rate-rise convention. Flip LONG_ON_DECREASE to switch.)
LONG_ON_DECREASE = True
# Threshold gate: trade only when |yhat| > X * train_pred_std. Default X=0 reproduces
# v5 Part C's always-on sign export; raw yhat + train_pred_std are emitted so the
# downstream sim can sweep X without re-fitting.
DEFAULT_X = 0.0

OUT_METRICS = 'mean_reversion_metrics_long.csv'
OUT_PRED    = 'mean_reversion_oos_predictions.csv'
V5_METRICS  = 'htboost_results_v5_nor/v5_metrics_long.csv'

print('Mean-reversion (OU/AR(1)) — v5-aligned harness')
print(f'  universe = {UNIVERSE}')
print(f'  horizons = {H_GRID}')
print(f'  folds    = {te.FOLD_NAMES}')
print(f'  config_hash = {CONFIG_HASH}   long_on_decrease={LONG_ON_DECREASE}  default_X={DEFAULT_X}')

In [ ]:
df_all = load_data()
print(f'Loaded {df_all.shape[0]} rows x {df_all.shape[1]} columns')
present = [s for s in UNIVERSE if s in df_all.columns]
missing = [s for s in UNIVERSE if s not in df_all.columns]
if missing:
    print('WARNING missing tenors:', missing)
print('\nHeadline universe coverage:')
for s in present:
    ser = df_all[s].dropna()
    print(f'  {s:<9s} n={len(ser):5d}  [{ser.index.min().date()} -> {ser.index.max().date()}]')

## Descriptive motivation (full sample) — NOT OOS evidence

ADF (H0: unit root), KPSS (H0: stationary) and the OU half-life on the **whole** sample,
shown only to motivate why a mean-reversion model is worth fitting. These are *not* used
for any selection and are *not* claimed as out-of-sample skill.

*Motivation (full sample only):* the unit-root / stationarity screen of Dickey & Fuller (1979) and Kwiatkowski et al. (1992) and the Ornstein–Uhlenbeck (1930) half-life motivate a Vasicek (1977) / Cox–Ingersoll–Ross (1985) mean-reverting specification.

In [ ]:
def adf_test(s):
    s = s.dropna()
    if len(s) < 30: return dict(adf_stat=np.nan, adf_p=np.nan, adf_reject=np.nan)
    r = adfuller(s, autolag='AIC')
    return dict(adf_stat=round(r[0],3), adf_p=round(r[1],4), adf_reject=bool(r[1]<0.05))

def kpss_test(s):
    s = s.dropna()
    if len(s) < 30: return dict(kpss_stat=np.nan, kpss_p=np.nan, kpss_reject=np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); r = kpss(s, regression='c', nlags='auto')
    return dict(kpss_stat=round(r[0],3), kpss_p=round(r[1],4), kpss_reject=bool(r[1]<0.05))

def ou_half_life(s):
    s = s.dropna()
    if len(s) < 30: return np.nan
    lag = s.shift(1).dropna(); delta = s.diff().dropna()
    idx = lag.index.intersection(delta.index)
    X = np.column_stack([np.ones(len(idx)), lag.loc[idx].values])
    beta = np.linalg.lstsq(X, delta.loc[idx].values, rcond=None)[0][1]
    return round(np.log(2)/abs(beta),1) if beta < 0 else np.nan

rows = []
for s in present:
    ser = df_all[s].dropna().sort_index()
    rows.append({'Series': s, 'n_obs': len(ser), **adf_test(ser), **kpss_test(ser),
                 'ou_half_life_days': ou_half_life(ser)})
desc_df = pd.DataFrame(rows).set_index('Series')
print('Descriptive stationarity (full sample, motivation only):')
display(desc_df)

## Continuous OU/AR(1) forecast — walk-forward, train + OOS

For each `(security, horizon, fold)` we refit a daily AR(1) on the fold's **training**
levels only:

```
level_{t+1} = c + phi * level_t      (OLS on train)
mu  = c / (1 - phi)                  (long-run level)
```

and form the **h-day-ahead expected change** (the OU point forecast):

```
yhat_h(t) = (mu - level_t) * (1 - phi^h)
```

a mean-reverting pull toward the train-estimated long-run level. If the AR(1) is
non-stationary (`phi` not in (0,1)) the model reduces to the random walk (`yhat = 0`).
One TRAIN row and one OOS row per populated fold are scored with `compute_metrics_row`.

*Model spec:* the AR(1)/Ornstein–Uhlenbeck pull toward a train-estimated long-run level operationalises Vasicek (1977) and Cox, Ingersoll & Ross (1985); the predictability it targets is the term-structure evidence of Fama & Bliss (1987), Campbell & Shiller (1991) and Cochrane & Piazzesi (2005).

In [ ]:
def build_panel(df, sec, h):
    level = df[sec].dropna().sort_index()
    y = level.shift(-h) - level
    return pd.DataFrame({'date': level.index, 'level': level.values, 'y': y.values})

def fit_ou(levels):
    """OLS daily AR(1) on a contiguous level array -> (phi, mu) or (None, None)."""
    x = np.asarray(levels, float)
    x = x[np.isfinite(x)]
    if x.size < 30:
        return None, None
    x0, x1 = x[:-1], x[1:]
    phi, c = np.polyfit(x0, x1, 1)          # x1 = c + phi*x0
    if not (0.0 < phi < 1.0):
        return None, None                   # non-reverting -> RW fallback
    return float(phi), float(c / (1.0 - phi))

def ou_forecast_change(level_t, phi, mu, h):
    if phi is None:
        return np.zeros_like(np.asarray(level_t, float))   # RW: forecast change = 0
    return (mu - np.asarray(level_t, float)) * (1.0 - phi ** h)

def run_security_mr(df, sec, h):
    """Walk-forward OU/AR(1) for one tenor at horizon h.
    Returns (metric_rows, oos_prediction_rows)."""
    country, tenor = sec.rsplit('_', 1)
    panel = build_panel(df, sec, h)
    base = dict(notebook=NOTEBOOK, run_ts=RUN_TS, model_kind=MODEL_KIND, is_pooled=False,
                validation_scheme='walk_forward', target_kind='level_change',
                security=sec, country=country, tenor=tenor,
                config_hash=CONFIG_HASH, feature_count=FEATURE_COUNT)
    rows, preds = [], []
    for fold_name, ts, tend, regime in WF_FOLDS:
        tr, tef = te.wf_purge_split(panel, ts, tend, h)
        if len(tr) < MIN_TRAIN_OBS or len(tef) < MIN_TEST_OBS:
            continue
        phi, mu = fit_ou(tr['level'].to_numpy(float))
        yhat_tr = ou_forecast_change(tr['level'].to_numpy(float), phi, mu, h)
        yhat_te = ou_forecast_change(tef['level'].to_numpy(float), phi, mu, h)
        # train_pred_std is computed from TRAIN-fold predictions ONLY (no OOS leakage)
        train_pred_std = float(np.std(yhat_tr[np.isfinite(yhat_tr)])) if np.isfinite(yhat_tr).any() else 0.0
        meta = {**base, 'fold': fold_name, 'regime': regime}
        rows.append(te.compute_metrics_row(tr['y'].to_numpy(float), yhat_tr, h, {**meta, 'sample': 'train'}))
        rows.append(te.compute_metrics_row(tef['y'].to_numpy(float), yhat_te, h, {**meta, 'sample': 'oos'}))
        # causal OOS trading rows (one per test date)
        sub = pd.DataFrame({'date': tef['date'].values, 'security': sec, 'country': country,
                            'tenor': tenor, 'horizon': h, 'regime': regime,
                            'level_t': tef['level'].values,
                            'y_true': tef['y'].values, 'yhat': yhat_te,
                            'train_pred_std': train_pred_std})
        preds.append(sub)
    return rows, preds

print('Model defined: fit_ou, ou_forecast_change, run_security_mr')

In [ ]:
metric_rows, pred_parts = [], []
for h in H_GRID:
    for sec in present:
        r, p = run_security_mr(df_all, sec, h)
        metric_rows.extend(r); pred_parts.extend(p)
        oos = [x for x in r if x['sample'] == 'oos']
        da = ', '.join(f"{x['dir_acc']:.3f}" for x in oos)
        print(f'  H={h:<3d} {sec:<9s} folds={len(oos)}  OOS_DA=[{da}]')

df_metrics = pd.DataFrame(metric_rows)
print(f'\ndf_metrics: {df_metrics.shape}  (train+oos rows)')

## Out-of-sample evaluation & multiple testing

Every fold is scored against a random-walk benchmark (`ŷ_RW = 0`) with the Campbell–Thompson
(2008) OOS-R², the Clark–West (2007) and Diebold–Mariano (1995) / Harvey–Leybourne–Newbold
(1997) equal-predictive-accuracy tests (HAC/Newey–West, lag `H−1`), the Pesaran–Timmermann
(1992) directional test, and a one-sided binomial. The walk-forward `{horizon × tenor ×
regime}` family is corrected with Bonferroni and BH-FDR (Benjamini & Hochberg, 1995; the
multiple-testing-in-finance critique of Harvey, Liu & Zhu, 2016). Walk-forward validation
follows the time-series cross-validation guidance of Bergmeir, Hyndman & Koo (2018) and the
purge/embargo discipline of López de Prado (2018).

In [ ]:
# ── Multiple-testing correction within the walk-forward family (v5 convention) ──
df_metrics = te.finalize_long_csv(df_metrics)
N, nb, nh = te.apply_mtc_family(df_metrics, ['walk_forward'], te.WF_MTC_FAMILY)
df_metrics = te.finalize_long_csv(df_metrics)   # re-order after MTC cols populated
df_metrics.to_csv(OUT_METRICS, index=False)
print(f'Wrote {OUT_METRICS}: rows={len(df_metrics)} cols={df_metrics.shape[1]}')
print(f'  walk-forward MTC family: N={N}  Bonferroni survivors={nb}  BH-FDR={nh}')

## Trading-sim export

The continuous OU/AR(1) forecast is turned into a position with a contrarian
mean-reversion sign convention (Poterba & Summers, 1988; Balvers, Wu & Gilliland, 2000):
raw `yhat` and the **train-only** `train_pred_std` are emitted so the downstream simulator
can sweep the entry threshold without re-fitting. Carry/roll is **not** fabricated here — the
simulator joins it from the shared curve layer using `level_t` + `tenor`.

In [ ]:
# ── Trading-sim export: causal walk-forward OOS predictions only (STEP 5) ────
pred = pd.concat(pred_parts, ignore_index=True)
pred = pred[np.isfinite(pred['level_t'])].reset_index(drop=True)   # drop missing level, never zero-fill

def positions_from(yhat, train_std, X=DEFAULT_X, long_on_decrease=LONG_ON_DECREASE):
    yhat = np.asarray(yhat, float)
    thr = X * np.asarray(train_std, float)
    direction = -1.0 if long_on_decrease else 1.0   # predicted decrease (yhat<0) -> +1
    return np.where(np.abs(yhat) > thr, direction * np.sign(yhat), 0.0)

pred['position'] = positions_from(pred['yhat'].values, pred['train_pred_std'].values)
pred = pred[['date', 'security', 'country', 'tenor', 'horizon', 'regime',
             'level_t', 'y_true', 'yhat', 'train_pred_std', 'position']]
pred = pred.sort_values(['horizon', 'date', 'security']).reset_index(drop=True)
pred.to_csv(OUT_PRED, index=False)
print(f'Wrote {OUT_PRED}: {len(pred)} OOS rows')
print('  position mix:', pred['position'].value_counts().to_dict())
print(pred.head(6).to_string(index=False))

In [ ]:
# ── Summary: OOS directional accuracy + Campbell-Thompson R2 by regime & horizon ──
oos = df_metrics[df_metrics['sample'] == 'oos'].copy()
print('=== Mean OOS directional accuracy  (regime x horizon) ===')
display(oos.pivot_table(index='regime', columns='horizon', values='dir_acc', aggfunc='mean').round(3))
print('=== Mean OOS Campbell-Thompson R2  (regime x horizon) ===')
display(oos.pivot_table(index='regime', columns='horizon', values='ct_r2_oos', aggfunc='mean').round(4))
print('=== Mean OOS n_eff (approx n/H)  (regime x horizon) — read dir_acc significance against this ===')
display(oos.pivot_table(index='regime', columns='horizon', values='n_eff', aggfunc='mean').round(0))
print('  (overlapping h-day labels inflate the binomial / Pesaran-Timmermann tests: judge them on n_eff, not n)')

n_cells = len(oos)
n_bonf = int(oos['reject_bonferroni'].sum())
n_fdr  = int(oos['reject_fdr_bh'].sum())
print(f'(security x horizon x regime) OOS cells: {n_cells}')
print(f'  survive Bonferroni: {n_bonf}    survive BH-FDR: {n_fdr}')

print('\n=== Train vs OOS dir_acc by horizon (overfit gap) ===')
gap = df_metrics.pivot_table(index='horizon', columns='sample', values='dir_acc', aggfunc='mean')
gap['gap(train-oos)'] = gap.get('train') - gap.get('oos')
display(gap.round(3))

In [ ]:
# ── Merge contract: must concatenate with v5_metrics_long.csv, zero reconciliation ──
v5 = pd.read_csv(V5_METRICS)
shared = te.SHARED_COLS
miss_here = [c for c in shared if c not in df_metrics.columns]
miss_v5   = [c for c in shared if c not in v5.columns]
assert not miss_here, f'missing shared cols here: {miss_here}'
assert not miss_v5,   f'missing shared cols in v5: {miss_v5}'
merged = pd.concat([v5, df_metrics], ignore_index=True)
assert all(c in merged.columns for c in shared), 'shared columns lost on concat'
print('MERGE OK — shared schema identical, no renames/drops.')
print(f'  v5 rows={len(v5)}  +  mean_reversion rows={len(df_metrics)}  =  merged {len(merged)}')
print(f'  shared cols={len(shared)}  merged cols={merged.shape[1]}')
print('  notebooks in merged:', sorted(merged["notebook"].unique()))

## Full universe of swaps — same pipeline, separate multiple-testing family

A **separate** section from the NOR head-to-head above. It runs the *identical* causal
walk-forward pipeline (`run_security_mr`, unchanged) on every available `country × tenor`
with enough history, writes **separate** files (`mean_reversion_metrics_long_universe.csv`,
`mean_reversion_oos_predictions_universe.csv`), and applies its **own** multiple-testing
family (the universe grid) so it can never contaminate the NOR family. Cross-market breadth
follows Balvers, Wu & Gilliland (2000); the wider grid makes the Harvey, Liu & Zhu (2016)
data-snooping correction more, not less, binding. **The NOR numbers above are unchanged by
anything in this section.**

In [ ]:
# (Universe a) definition — all swap prefixes × te tenor suffixes present with enough history
SWAP_PREFIXES  = ['AUS','BRAZ','CAN','CHIN','EUR','JPY','NEWZ','NOR','POL','SOFR','SONIA','SWE','SWZ','TURK']
TENOR_SUFFIXES = [t.split('_', 1)[1] for t in te.NOR_TENORS]      # 1Y, 5Y, 10Y, 15Y, 30Y
MIN_UNIV_OBS   = te.MIN_TRAIN_OBS + te.MIN_TEST_OBS               # same gate as a scored fold pair
universe = [f'{p}_{m}' for p in SWAP_PREFIXES for m in TENOR_SUFFIXES
            if f'{p}_{m}' in df_all.columns
            and df_all[f'{p}_{m}'].dropna().shape[0] >= MIN_UNIV_OBS]
print(f'Universe: {len(universe)} securities across {len(SWAP_PREFIXES)} markets x {TENOR_SUFFIXES}')
print(f'  inclusion gate: >= MIN_TRAIN_OBS + MIN_TEST_OBS = {MIN_UNIV_OBS} obs')
print(' ', universe)

In [ ]:
# (Universe b) descriptive motivation screen — reuse the functions defined above (no redefine)
urows = []
for c in universe:
    s = df_all[c].dropna().sort_index()
    urows.append({'Series': c, 'n_obs': len(s), **adf_test(s), **kpss_test(s),
                  'ou_half_life_days': ou_half_life(s)})
univ_desc = pd.DataFrame(urows).set_index('Series')
univ_desc['mean_reverting'] = (univ_desc['adf_reject'] == True) & (univ_desc['kpss_reject'] == False)
print(f'Universe descriptive screen ({len(univ_desc)} series, motivation only); '
      f"mean-reverting evidence: {int(univ_desc['mean_reverting'].sum())}")
display(univ_desc.sort_values('adf_p'))

In [ ]:
# (Universe c) causal walk-forward sweep — reuse run_security_mr UNCHANGED over the universe
univ_rows, univ_preds = [], []
for h in H_GRID:
    for sec in universe:
        r, p = run_security_mr(df_all, sec, h)
        univ_rows.extend(r); univ_preds.extend(p)
df_univ = pd.DataFrame(univ_rows)
print(f'Universe sweep: df_univ {df_univ.shape}  ({len(universe)} secs x {len(H_GRID)} horizons)')

In [ ]:
# (Universe d) finalize + SEPARATE MTC family (the universe grid) + breakdowns + write CSV
OUT_METRICS_UNIV = 'mean_reversion_metrics_long_universe.csv'
UNIV_MTC_FAMILY  = 'walk_forward:universe:{country x tenor x horizon x regime}'
df_univ = te.finalize_long_csv(df_univ)
Nu, nbu, nhu = te.apply_mtc_family(df_univ, ['walk_forward'], UNIV_MTC_FAMILY)
df_univ = te.finalize_long_csv(df_univ)
df_univ.to_csv(OUT_METRICS_UNIV, index=False)
print(f'Wrote {OUT_METRICS_UNIV}: rows={len(df_univ)} cols={df_univ.shape[1]}')
print(f'  universe MTC family N={Nu}  Bonferroni survivors={nbu}  BH-FDR={nhu}  '
      f'(SEPARATE family from NOR)')

uoos = df_univ[df_univ['sample'] == 'oos'].copy()
print('\n=== Universe mean OOS dir_acc  (regime x horizon) ===')
display(uoos.pivot_table(index='regime', columns='horizon', values='dir_acc', aggfunc='mean').round(3))
print('=== Universe mean OOS Campbell-Thompson R2  (regime x horizon) ===')
display(uoos.pivot_table(index='regime', columns='horizon', values='ct_r2_oos', aggfunc='mean').round(4))
print('=== Universe mean OOS dir_acc  (country x horizon) ===')
display(uoos.pivot_table(index='country', columns='horizon', values='dir_acc', aggfunc='mean').round(3))
print('=== Universe mean OOS Campbell-Thompson R2  (country x horizon) ===')
display(uoos.pivot_table(index='country', columns='horizon', values='ct_r2_oos', aggfunc='mean').round(4))
print('=== Universe mean OOS n_eff (approx n/H)  (regime x horizon) — read dir_acc significance against this ===')
display(uoos.pivot_table(index='regime', columns='horizon', values='n_eff', aggfunc='mean').round(0))
print(f'\nUniverse OOS cells: {len(uoos)}   survive Bonferroni: {int(uoos["reject_bonferroni"].sum())}'
      f'   survive BH-FDR: {int(uoos["reject_fdr_bh"].sum())}   (family N={Nu})')

In [ ]:
# (Universe e) trading export — RUN AFTER the tests above; identical schema to the NOR export
OUT_PRED_UNIV = 'mean_reversion_oos_predictions_universe.csv'
upred = pd.concat(univ_preds, ignore_index=True)
upred = upred[np.isfinite(upred['level_t'])].reset_index(drop=True)   # drop missing level, never zero-fill
upred['position'] = positions_from(upred['yhat'].values, upred['train_pred_std'].values)
upred = upred[['date', 'security', 'country', 'tenor', 'horizon', 'regime',
               'level_t', 'y_true', 'yhat', 'train_pred_std', 'position']]
upred = upred.sort_values(['horizon', 'date', 'security']).reset_index(drop=True)
upred.to_csv(OUT_PRED_UNIV, index=False)
print(f'Wrote {OUT_PRED_UNIV}: {len(upred)} OOS rows')
assert list(upred.columns) == list(pred.columns), 'universe export schema must equal the NOR export schema'
print('  schema identical to NOR export:', list(upred.columns))
print('  position mix:', upred['position'].value_counts().to_dict())

## Appendix — legacy rule-based ±1 (secondary cross-check)

Not part of the head-to-head and **not** written to the shared CSV (the ±1 signal has no magnitude). The full-universe descriptive screen now lives in the *Full universe of swaps* section above; what remains here is the legacy rule-based z-score ±1 signal scored on the same walk-forward folds for the NOR tenors — directional accuracy on the **full** OOS sample, with coverage as an auxiliary column.

In [ ]:
# (b) legacy rule-based z-score +/-1 on NOR tenors, walk-forward, OOS dir_acc (full sample)
def zscore_signal(level, window, threshold):
    ma = level.rolling(window, min_periods=window).mean()
    sd = level.rolling(window, min_periods=window).std()
    z = (level - ma) / sd
    sig = pd.Series(0.0, index=level.index)
    sig[z >  threshold] = -1.0   # stretched above MA -> expect fall
    sig[z < -threshold] =  1.0   # stretched below MA -> expect rise
    return sig

RULE_WINDOW, RULE_THRESHOLD = 60, 1.0   # pre-committed (no OOS tuning)
print(f'Rule-based z-score +/-1 (window={RULE_WINDOW}, thr={RULE_THRESHOLD}) — SECONDARY, not in CSV')
brows = []
for h in H_GRID:
    for sec in present:
        level = df_all[sec].dropna().sort_index()
        sig = zscore_signal(level, RULE_WINDOW, RULE_THRESHOLD)
        panel = build_panel(df_all, sec, h).set_index('date')
        for fold_name, ts, tend, regime in WF_FOLDS:
            _, tef = te.wf_purge_split(panel.reset_index(), ts, tend, h)
            if len(tef) < MIN_TEST_OBS: continue
            d = tef.set_index('date')
            s = sig.reindex(d.index).fillna(0.0)
            y = d['y']
            m = np.isfinite(y)
            da_full = float(np.mean(np.sign(s[m]) == np.sign(y[m])))   # full OOS sample
            cov = float(np.mean(s[m] != 0))
            brows.append(dict(horizon=h, security=sec, regime=regime,
                              dir_acc_full=round(da_full,3), coverage=round(cov,3), n=int(m.sum())))
rule_df = pd.DataFrame(brows)
print('Mean OOS dir_acc (full sample) by regime x horizon:')
display(rule_df.pivot_table(index='regime', columns='horizon', values='dir_acc_full', aggfunc='mean').round(3))